In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import LabelEncoder
import joblib
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 1. Load the single, entire dataset
full_data = pd.read_csv('/content/dataset.csv')

In [ ]:
# 2. Clean up (Drop coordinates but KEEP SystemCodeNumber)
full_data = full_data.drop(columns=['ID', 'Latitude', 'Longitude'])
full_data = full_data[~(full_data['Occupancy'] > full_data['Capacity'])]

In [ ]:
# 3. Combine Date and Time
full_data['Timestamp'] = pd.to_datetime(full_data['LastUpdatedDate'] + ' ' + full_data['LastUpdatedTime'], format='%d-%m-%Y %H:%M:%S')
full_data.drop(columns=['LastUpdatedDate','LastUpdatedTime'], inplace=True)

In [ ]:
# 4. Sort chronologically by Lot AND Time (Crucial for target shifting)
full_data = full_data.sort_values(by=['SystemCodeNumber', 'Timestamp']).reset_index(drop=True)


In [ ]:
full_data.info();

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18127 entries, 0 to 18126
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   SystemCodeNumber        18127 non-null  object        
 1   Capacity                18127 non-null  int64         
 2   Occupancy               18127 non-null  int64         
 3   VehicleType             18127 non-null  object        
 4   TrafficConditionNearby  18127 non-null  object        
 5   QueueLength             18127 non-null  int64         
 6   IsSpecialDay            18127 non-null  int64         
 7   Timestamp               18127 non-null  datetime64[ns]
dtypes: datetime64[ns](1), int64(4), object(3)
memory usage: 1.1+ MB


In [ ]:
full_data.describe();

In [ ]:
# 5. Extract Temporal Features
full_data['Hour'] = full_data['Timestamp'].dt.hour
full_data['DayOfWeek'] = full_data['Timestamp'].dt.dayofweek


In [ ]:
# 6. Encode Categorical Variables
traffic_map = {'low': 3.0, 'average': 6.0, 'high': 9.0}
full_data['traffic_level'] = full_data['TrafficConditionNearby'].map(traffic_map)


In [ ]:
vehicle_map = {'cycle': 0.65, 'bike': 0.85, 'car': 1.15, 'truck': 1.35}
full_data['vehicle_type_weight'] = full_data['VehicleType'].map(vehicle_map)


In [ ]:
# ENCODE THE PARKING LOT ID
# This allows the single model to differentiate between the 14 lots
lot_encoder = LabelEncoder()
full_data['Lot_ID'] = lot_encoder.fit_transform(full_data['SystemCodeNumber'])


In [ ]:
# 7. Shift the Target (Predict 30 mins into the future)
steps_to_shift = -6
full_data['Future_Occupancy'] = full_data.groupby('SystemCodeNumber')['Occupancy'].shift(steps_to_shift)
full_data = full_data.dropna(subset=['Future_Occupancy'])


In [ ]:
# 8. Define the updated Features (X) including Lot_ID
features = [
    'Lot_ID', 'Capacity', 'QueueLength', 'IsSpecialDay',
    'traffic_level', 'vehicle_type_weight', 'Hour', 'DayOfWeek'
]
X = full_data[features]
y = full_data['Future_Occupancy']

In [ ]:
# 9. Time-Series Split
split_index = int(len(full_data) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

In [ ]:
# 10. Train the XGBoost Model
print("Training XGBoost Model...")
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    objective='reg:squarederror'
)
xgb_model.fit(X_train, y_train)

Training XGBoost Model...


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
# Evaluate
predictions = xgb_model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
print(f"Model Mean Absolute Error: {mae:.2f} cars")

Model Mean Absolute Error: 243.43 cars


In [ ]:
# 11. SAVE THE MODELS
joblib.dump(xgb_model, 'parking_pricing_model.pkl')
joblib.dump(lot_encoder, 'lot_encoder.pkl')

['lot_encoder.pkl']

In [ ]:
print("Ready for Time-Series Split and Model Training!")
print(X.head())


Ready for Time-Series Split and Model Training!
   Lot_ID  Capacity  QueueLength  IsSpecialDay  traffic_level  \
0       0       577            1             0            3.0   
1       0       577            1             0            3.0   
2       0       577            2             0            3.0   
3       0       577            2             0            3.0   
4       0       577            2             0            3.0   

   vehicle_type_weight  Hour  DayOfWeek  
0                 1.15     7          1  
1                 1.15     8          1  
2                 1.15     8          1  
3                 1.15     9          1  
4                 0.85     9          1  
